# Simple Federated Learning (FL) on MNIST

This notebook demonstrates a minimal, educational Federated Learning pipeline implemented from scratch in PyTorch. It trains a small MLP centrally (baseline) and then simulates multiple clients that perform local training and Federated Averaging. The notebook is intentionally lightweight so it runs quickly on a CPU.

## Imports

In [2]:
import copy
import math
import random
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

# Reproducibility
random.seed(0)
torch.manual_seed(0)

## Dataset loading

In [ ]:
# On entraîne le modèle par petits groupes de 64 images
BATCH_SIZE = 64
NUM_WORKERS = 0  # pour l'instant

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# MNIST transforms: ToTensor + Normalize (0.1307 mean, 0.3081 std commonly used)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

# Split entre train et test
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader_full = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print('Train samples:', len(train_dataset), 'Test samples:', len(test_dataset))

Device: cpu


100.0%
100.0%
100.0%
100.0%

Train samples: 60000 Test samples: 10000


Dans cette partie, je charge le dataset MNIST et je prépare les données pour l’entraînement.

Les données sont transformées en tenseurs et normalisées afin d’améliorer les performances du modèle.

Ensuite, j’utilise des DataLoaders pour traiter les données par batch, ce qui permet un entraînement plus efficace.

## Le modèle (SimpleMLP)

In [17]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten() #transformation vers un vecteur
        self.fc1 = nn.Linear(28 * 28, 128) # 1ere couche
        self.fc2 = nn.Linear(128, 64) # 2eme couche
        self.out = nn.Linear(64, 10) # couche de sortie 

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x

# nouveau modèle pour chaque client
def create_model():
    return SimpleMLP()

J’utilise un réseau de neurones de type MLP.

L’image est d’abord transformée en vecteur, puis elle passe à travers plusieurs couches linéaires avec des fonctions d’activation ReLU.

La dernière couche produit une sortie de dimension 10 correspondant aux classes possibles.

## Entraînement, Évaluation et Split

In [ ]:
def evaluate(model: nn.Module, dataloader: DataLoader, device: torch.device) -> Tuple[float, float]:
    """Return (loss, accuracy) on the provided dataloader.
    Model is expected to be in eval() mode when called.
    """
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='sum')
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

def train_one_epoch(model: nn.Module, dataloader: DataLoader, optimizer, device: torch.device) -> None:
    model.to(device)
    model.train()
    criterion = nn.CrossEntropyLoss()
    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

def split_dataset_iid(dataset, num_clients: int) -> List[Subset]:
    """fonction pour diviser un ensemble de données en sous-ensembles IID pour
    chaque client
    """
    total_len = len(dataset)
    base = total_len // num_clients
    lengths = [base] * num_clients
    rem = total_len - base * num_clients
    # distribuer les éléments restants (s'il y en a) aux premiers clients
    for i in range(rem):
        lengths[i] += 1
    # créer des sous-ensembles
    subsets = random_split(dataset, lengths)
    return list(subsets)

## Résumé des fonctions

- **evaluate** : calcule la loss et l’accuracy du modèle (sans entraînement)
- **train_one_epoch** : entraîne le modèle sur une époque
- **split_dataset_iid** : divise les données en plusieurs clients (IID)

## Entraînement centralisé

In [ ]:
# nouveau modèle pour chaque client
central_model = create_model().to(device)
optimizer = torch.optim.SGD(central_model.parameters(), lr=0.1) 
CENTRAL_EPOCHS = 2  

for epoch in range(1, CENTRAL_EPOCHS + 1):
    train_one_epoch(central_model, train_loader_full, optimizer, device)
    loss, acc = evaluate(central_model, test_loader, device)
    print(f'Centralized Epoch {epoch}/{CENTRAL_EPOCHS} -> Test loss: {loss:.4f}, Test acc: {acc:.4f}')

central_loss, central_acc = evaluate(central_model, test_loader, device)
print(
    'Centralized baseline final -> Loss: {:.4f}, Acc: {:.4f}'.format(
        central_loss, central_acc
    )
)

Centralized Epoch 1/2 -> Test loss: 0.1702, Test acc: 0.9472
Centralized Epoch 2/2 -> Test loss: 0.0954, Test acc: 0.9700
Centralized Epoch 2/2 -> Test loss: 0.0954, Test acc: 0.9700
Centralized baseline final -> Loss: 0.0954, Acc: 0.9700
Centralized baseline final -> Loss: 0.0954, Acc: 0.9700




Dans cette étape, un modèle est entraîné de manière classique sur l’ensemble du dataset.

Le modèle est initialisé puis entraîné pendant plusieurs époques à l’aide d’un optimiseur (SGD).  
À chaque époque, les performances sont évaluées sur le jeu de test afin de suivre l’évolution de la loss et de l’accuracy.

Cette étape permet d’obtenir une **baseline de référence**, qui sera ensuite comparée aux performances du modèle entraîné en Federated Learning.

## Simulation des clients (IID split)

In [ ]:
NUM_CLIENTS = 5  
client_subsets = split_dataset_iid(train_dataset, NUM_CLIENTS)
client_loaders = [DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS) for subset in client_subsets]
client_sizes = [len(s) for s in client_subsets]
print('Client data sizes (IID):', client_sizes)

Client data sizes (IID): [12000, 12000, 12000, 12000, 12000]


## Local training (clients train for 1 epoch)

In [10]:
def local_train_update(global_state_dict, train_loader: DataLoader, device: torch.device, lr: float = 0.1, epochs: int = 1):
    """Create a local model, load the global weights, train locally, and return the updated state_dict and number of samples.
    """
    model = create_model()
    model.load_state_dict(global_state_dict)
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    for e in range(epochs):
        train_one_epoch(model, train_loader, optimizer, device)
    return model.state_dict(), len(train_loader.dataset)

# Quick smoke test: run local update on first client using central model state
test_state, test_samples = local_train_update(central_model.state_dict(), client_loaders[0], device, lr=0.1, epochs=1)
print('Local training produced state dict keys:', list(test_state.keys())[:5], 'num samples:', test_samples)

Local training produced state dict keys: ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'out.weight'] num samples: 12000


## Federated Averaging (weighted)

In [12]:
def federated_average(state_dicts: List[dict], weights: List[float]) -> dict:
    """Average a list of state_dicts using the provided weights (same keys assumed).
    weights are normalized to sum to 1.
    """
    # normalize weights
    total = float(sum(weights))
    if total == 0:
        raise ValueError('Sum of weights must be > 0')
    norm_weights = [w / total for w in weights]

    avg_state = {}
    for key in state_dicts[0].keys():
        # start from zeros of appropriate shape and dtype
        avg_state[key] = torch.zeros_like(state_dicts[0][key])
        for sd, w in zip(state_dicts, norm_weights):
            avg_state[key] += sd[key].float() * w
    # ensure same dtype as original params
    for key in avg_state.keys():
        avg_state[key] = avg_state[key].type(state_dicts[0][key].dtype)
    return avg_state

## Federated Learning loop

In [14]:
ROUNDS = 3  # between 3-5 rounds as requested
FED_LR = 0.1  # local learning rate for clients
global_model = create_model()
global_state = global_model.state_dict()

# Track accuracy per round
round_results = []
for r in range(1, ROUNDS + 1):
    print(f'--- Federated Round {r}/{ROUNDS} ---')
    local_states = []
    local_sizes = []
    # Each client trains locally for 1 epoch and returns its updated weights
    for i, loader in enumerate(client_loaders):
        sd, n_samples = local_train_update(global_state, loader, device, lr=FED_LR, epochs=1)
        local_states.append(sd)
        local_sizes.append(n_samples)
        print(f' Client {i+1}: trained on {n_samples} samples')
    # Aggregate
    new_global_state = federated_average(local_states, local_sizes)
    global_state = new_global_state
    global_model.load_state_dict(global_state)
    # Evaluate on test set
    loss, acc = evaluate(global_model, test_loader, device)
    round_results.append((loss, acc))
    print(f' Round {r} -> Test loss: {loss:.4f}, Test acc: {acc:.4f}')

--- Federated Round 1/3 ---
 Client 1: trained on 12000 samples
 Client 1: trained on 12000 samples
 Client 2: trained on 12000 samples
 Client 2: trained on 12000 samples
 Client 3: trained on 12000 samples
 Client 3: trained on 12000 samples
 Client 4: trained on 12000 samples
 Client 4: trained on 12000 samples
 Client 5: trained on 12000 samples
 Client 5: trained on 12000 samples
 Round 1 -> Test loss: 0.2911, Test acc: 0.9126
--- Federated Round 2/3 ---
 Round 1 -> Test loss: 0.2911, Test acc: 0.9126
--- Federated Round 2/3 ---
 Client 1: trained on 12000 samples
 Client 1: trained on 12000 samples
 Client 2: trained on 12000 samples
 Client 2: trained on 12000 samples
 Client 3: trained on 12000 samples
 Client 3: trained on 12000 samples
 Client 4: trained on 12000 samples
 Client 4: trained on 12000 samples
 Client 5: trained on 12000 samples
 Client 5: trained on 12000 samples
 Round 2 -> Test loss: 0.1952, Test acc: 0.9425
--- Federated Round 3/3 ---
 Round 2 -> Test loss: 0

## Evaluation and comparison

In [16]:
print('Centralized baseline -> Loss: {:.4f}, Acc: {:.4f}'.format(central_loss, central_acc))
for i, (loss, acc) in enumerate(round_results, 1):
    print(f'Federated Round {i}: Test Loss = {loss:.4f}, Test Acc = {acc:.4f}')

final_fed_loss, final_fed_acc = round_results[-1]
print('Summary:')
print(f' Centralized acc: {central_acc:.4f}')
print(f' Federated (after {ROUNDS} rounds) acc: {final_fed_acc:.4f}')

Centralized baseline -> Loss: 0.0954, Acc: 0.9700
Federated Round 1: Test Loss = 0.2911, Test Acc = 0.9126
Federated Round 2: Test Loss = 0.1952, Test Acc = 0.9425
Federated Round 3: Test Loss = 0.1631, Test Acc = 0.9488
Summary:
 Centralized acc: 0.9700
 Federated (after 3 rounds) acc: 0.9488


## Notes and next steps
- This is a minimal educational example: single-epoch local updates, small MLP, and a few rounds.
- For production or research you would add: more local epochs, learning rate schedules, secure aggregation, client selection, heterogeneous (non-IID) splits, and communication simulation.
- To run longer experiments, increase epochs and consider using GPU (if available).

: 4,
: 5